# Zarr Creation & Performance Benchmarking

## Introduction

To create the data cube in virtualzarr format using the outputs from `gdal` and `cdo`.

### Key Objectives:

1. **Visualize outputs** from CDO and GDAL preprocessing pipelines
2. **Create Zarr stores** from preprocessed netCDF files
3. **Benchmark performance** comparing:
   - NetCDF (original format)
   - Zarr (converted format)
4. **Measure metrics**: Load time, memory usage, file size, and query performance




In [1]:
# Import Required Libraries (NetCDF -> Zarr workflow only)

# Fix matplotlib backend issue from environment
import os
if "MPLBACKEND" in os.environ:
    del os.environ["MPLBACKEND"]

# Core analysis
import xarray as xr
import numpy as np
import pandas as pd

# Zarr + performance
import zarr
import time
import psutil
import gc

import lexcube

# Optional NetCDF backend import (some environments may not have it)
try:
    import netCDF4  # noqa: F401
except Exception as e:
    print(f"Note: netCDF4 import failed: {e}")

# Visualization
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams

# File & system ops
import glob
import json
from pathlib import Path
from datetime import datetime

# Environment knobs
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# Warnings & logging
import warnings
warnings.filterwarnings("ignore")

# Configure matplotlib defaults
rcParams["figure.figsize"] = (14, 8)
rcParams["font.size"] = 10
rcParams["axes.labelsize"] = 11
rcParams["axes.titlesize"] = 12
rcParams["xtick.labelsize"] = 9
rcParams["ytick.labelsize"] = 9
rcParams["legend.fontsize"] = 10
rcParams["figure.titlesize"] = 14

print("Imports OK")
print(f"  - xarray: {xr.__version__}")
print(f"  - numpy:  {np.__version__}")
print(f"  - pandas: {pd.__version__}")
print(f"  - zarr:   {zarr.__version__}")

Imports OK
  - xarray: 2025.11.0
  - numpy:  2.4.2
  - pandas: 3.0.0
  - zarr:   3.1.5


## Notebook Configuration

Before running the analysis, ensure you have the following dependencies installed:

```bash
pip install xarray zarr pandas matplotlib psutil netCDF4
```

This notebook uses data outputs from the CDO and GDAL preprocessing pipelines located in:
- `../outputs/cdo_output/` - CDO processed netCDF files
- `../outputs/gdal_output/` - GDAL processed netCDF files
- `../outputs/zarr_outputs/` - Generated Zarr stores (created by this notebook)


In [11]:
### 1. Define paths to CDO and GDAL outputs

# Setup directory paths
outputs_dir = "../outputs"
cdo_output_dir, gdal_output_dir = f"{outputs_dir}/cdo_output", f"{outputs_dir}/gdal_output"

# Define subdirectories
paths = {
    'cdo_metref': f"{cdo_output_dir}/metref",
    'cdo_rzsm': f"{cdo_output_dir}/rzsm",
    'gdal_metref': f"{gdal_output_dir}/metref",
    'gdal_sm': f"{gdal_output_dir}/rzsm"
}

# Find all files with corrected glob patterns
cdo_metref_files = sorted(glob.glob(f"{paths['cdo_metref']}/**/*.nc", recursive=True))
cdo_rzsm_files = sorted(glob.glob(f"{paths['cdo_rzsm']}/**/*.nc", recursive=True))
gdal_metref_files = sorted(glob.glob(f"{paths['gdal_metref']}/**/METREF/*.nc", recursive=True))

# GDAL SM files organized by variable folders (var40, var41, var42, var43)
gdal_sm_files = {}
for var_folder in ['var40', 'var41', 'var42', 'var43']:
    var_path = f"{paths['gdal_sm']}/**/{var_folder}/*.nc"
    files = sorted(glob.glob(var_path, recursive=True))
    if files:
        gdal_sm_files[var_folder] = files

# Print summary
print(f"CDO Output: {cdo_output_dir}\nGDAL Output: {gdal_output_dir}")
print(f"\nCDO METREF: {len(cdo_metref_files)} files | CDO RZSM: {len(cdo_rzsm_files)} files")
print(f"GDAL METREF: {len(gdal_metref_files)} files")
print(f"GDAL SM variables found: {list(gdal_sm_files.keys())}")
for var, files in gdal_sm_files.items():
    print(f"  {var}: {len(files)} files")


CDO Output: ../outputs/cdo_output
GDAL Output: ../outputs/gdal_output

CDO METREF: 5 files | CDO RZSM: 5 files
GDAL METREF: 5 files
GDAL SM variables found: ['var40', 'var41', 'var42', 'var43']
  var40: 5 files
  var41: 5 files
  var42: 5 files
  var43: 5 files


In [12]:
# Configure input NetCDF directory (CDO output example)

store_path = Path("/home/kzakir/dvcube/test/outputs/gdal_output/metref/2010/01").resolve()
assert store_path.exists(), f"Missing input folder: {store_path}"

# Collect NetCDF files
all_files = sorted(glob.glob(str(store_path / "**" / "*.nc"), recursive=True))
print(f"Found {len(all_files)} NetCDF files under {store_path}")
assert len(all_files) > 0, "No .nc files found"

# Preview first file path
print("First file:", all_files[0])

Found 10 NetCDF files under /home/kzakir/dvcube/test/outputs/gdal_output/metref/2010/01
First file: /home/kzakir/dvcube/test/outputs/gdal_output/metref/2010/01/METREF/NETCDF4_LSASAF_MSG_METREF_MSG-Disk_201001010000_METREF_processed.nc


In [13]:
# Open multiple files with xarray

# If you get backend errors with netCDF4, switch to engine="h5netcdf" here too.
ds_nc = xr.open_mfdataset(
    all_files,
    combine="nested",
    concat_dim="time"
)

ds_nc

<xarray.Dataset> Size: 48MB
Dimensions:       (time: 10, lat: 200, lon: 1500)
Coordinates:
  * lat           (lat) float64 2kB 10.03 10.07 10.12 ... 19.88 19.93 19.98
  * lon           (lon) float64 12kB -19.98 -19.93 -19.88 ... 54.88 54.92 54.98
Dimensions without coordinates: time
Data variables:
    METREF        (time, lat, lon) float64 24MB dask.array<chunksize=(1, 200, 1500), meta=np.ndarray>
    crs           (time) |S1 10B b'' b'' b'' b'' b'' b'' b'' b'' b'' b''
    quality_flag  (time, lat, lon) float64 24MB dask.array<chunksize=(1, 200, 1500), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.5
    GDAL:         GDAL 3.4.1, released 2021/12/27
    history:      Thu Feb 12 09:37:25 2026: ncrename -v Band1,METREF /home/kz...
    NCO:          netCDF Operators version 5.0.6 (Homepage = http://nco.sf.ne...

## Make Zarr format from the combine files

In [14]:
# Write combined dataset directly to Zarr

import shutil

zarr_store_path = Path("../outputs/zarr_outputs/metref_combined_xarray_gdal.zarr").resolve()
zarr_store_path.parent.mkdir(parents=True, exist_ok=True)
if zarr_store_path.exists():
    shutil.rmtree(zarr_store_path)

t0 = time.time()

# 1) Chunk the combined dataset
chunk_plan = {"time": 1}
for dim in ["lat", "latitude", "y"]:
    if dim in ds_nc.dims:
        chunk_plan[dim] = 256
for dim in ["lon", "longitude", "x"]:
    if dim in ds_nc.dims:
        chunk_plan[dim] = 256

ds_chunked = ds_nc.chunk(chunk_plan)

# 2) Write to Zarr
ds_chunked.to_zarr(zarr_store_path, mode="w", consolidated=True)

dt = time.time() - t0
print(f"✓ Wrote Zarr store: {zarr_store_path}")
print(f"  - wall time: {dt:.2f}s")

✓ Wrote Zarr store: /home/kzakir/dvcube/test/outputs/zarr_outputs/metref_combined_xarray_gdal.zarr
  - wall time: 0.44s


In [38]:
# 3) Reopen with xr.open_zarr and read a sample (sanity check)
zarr_store_path = Path("../outputs/zarr_outputs/metref_combined_xarray.zarr").resolve()

ds_z = xr.open_zarr(zarr_store_path, consolidated=True)
print("\nZarr dataset:")
print(ds_z)

varname = "METREF" if "METREF" in ds_z.data_vars else list(ds_z.data_vars)[0]
print(f"\nUsing variable: {varname}")

# Try scalar at (0,0,0)
scalar_indexers = {"time": 0}
if "lat" in ds_z.dims:
    scalar_indexers["lat"] = 0
if "lon" in ds_z.dims:
    scalar_indexers["lon"] = 0

val = ds_z[varname].isel(**scalar_indexers).load()
print(f"Scalar at (time=0, lat=0, lon=0): {float(val.values) if np.ndim(val.values) == 0 else val.values}")

# # Try a small tile to see if there are valid values elsewhere
# print("\n--- Reading a 10x10 tile (time=0, lat=100:110, lon=100:110) ---")
# try:
#     tile_indexers = {"time": 0}
#     if "lat" in ds_z.dims:
#         tile_indexers["lat"] = slice(100, 110)
#     if "lon" in ds_z.dims:
#         tile_indexers["lon"] = slice(100, 110)
    
#     tile = ds_z[varname].isel(**tile_indexers).load()
#     print(f"Tile shape: {tile.shape}")
#     print(f"Tile dtype: {tile.dtype}")
#     print(f"Tile min: {np.nanmin(tile.values):.4f}")
#     print(f"Tile max: {np.nanmax(tile.values):.4f}")
#     print(f"Valid (non-NaN) count: {np.sum(~np.isnan(tile.values))} / {tile.size}")
# except Exception as e:
#     print(f"Tile read failed: {e}")


Zarr dataset:
<xarray.Dataset> Size: 24MB
Dimensions:       (time: 5, lat: 200, lon: 1500)
Coordinates:
  * time          (time) datetime64[ns] 40B 2010-01-01 2010-01-02 ... 2010-01-05
  * lat           (lat) float64 2kB 10.0 10.05 10.1 10.15 ... 19.85 19.9 19.95
  * lon           (lon) float64 12kB -20.0 -19.95 -19.9 ... 54.85 54.9 54.95
Data variables:
    METREF        (time, lat, lon) float64 12MB dask.array<chunksize=(1, 200, 256), meta=np.ndarray>
    quality_flag  (time, lat, lon) float64 12MB dask.array<chunksize=(1, 200, 256), meta=np.ndarray>
Attributes: (12/29)
    CDI:                        Climate Data Interface version 2.0.4 (https:/...
    Conventions:                CF-1.6
    institution:                IM-PT
    date_created:               2020-11-09T17:30:27Z
    algorithm_version:          1.3.2
    base_algorithm_version:     1.0.3
    ...                         ...
    westernmost_longitude:      80.0
    spatial_resolution:          0.05x 0.05
    geospatial_l

In [39]:
#plot

ds_z = ds_z.rio.write_crs("EPSG:4326")

In [40]:
# !pip install lexcube

In [50]:
da = ds_z['METREF'][:,:,:]
w = lexcube.Cube3DWidget(da, cmap="magma_r", vmin = 1, vmax = 4)
w

Cube3DWidget(api_metadata={'/api': {'status': 'ok', 'api_version': 5}, '/api/datasets': [{'id': 'default', 'sh…

In [22]:
w.show_sliders()

Sliders(children=(HBox(children=(IntRangeSlider(value=(0, 1499), description='lon:', max=1499), Label(value='-…

In [47]:
import geopandas as gpd
gdf = gpd.read_file("/home/kzakir/dvcube/test/cube_example/sah_admin0_ocha/sah_admin0_ocha.shp")
gdf.to_crs('epsg:4326')
gdf.to_file("sah_admin_0ocha.geosjon", driver="GeoJSON")


In [48]:
w.overlay_geojson("sah_admin_0ocha.geosjon")

Opening GeoJSON from file...
Loaded GeoJSON from file sah_admin_0ocha.geosjon.


In [30]:
gdf.head
gdf.plot

In [53]:
w.savefig(include_ui=False)

When using Lexcube and generated images or videos, please acknowledge/cite: Söchting, M., Scheuermann, G., Montero, D., & Mahecha, M. D. (2025). Interactive Earth system data cube visualization in Jupyter notebooks. Big Earth Data, 1–15. https://doi.org/10.1080/20964471.2025.2471646
